# DAG – Honda RA621H Assembly
**Struktur Data dan Algoritma | Proyek 2025**

Notebook ini membangun Directed Acyclic Graph (DAG) dari dataset perakitan Honda RA621H.
- 103 nodes (komponen)
- 216 directed edges (dependency)
- 16 root nodes (tidak punya predecessor)

In [ ]:
# Cell 0 – Import library
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

print('Library siap.')

In [ ]:
# Cell 1 – Load dataset
df_comp = pd.read_excel('Honda_RA621H_Assembly_Dataset.xlsx', sheet_name='Components', header=2)
df_edge = pd.read_excel('Honda_RA621H_Assembly_Dataset.xlsx', sheet_name='Dependency_Edges', header=2)

# Bersihkan kolom yang relevan
df_comp = df_comp[['Component ID', 'Component Name', 'Subsystem',
                    'Depends On (IDs)', 'Duration (hrs)', 'Is Critical Path']].copy()
df_comp['is_critical'] = df_comp['Is Critical Path'].str.contains('CRITICAL', na=False)

print(f'Komponen (nodes) : {len(df_comp)}')
print(f'Edges            : {len(df_edge)}')
df_comp.head()

In [ ]:
# Cell 2 – Bangun DAG dengan NetworkX
G = nx.DiGraph()

# Tambah nodes beserta atributnya
for _, row in df_comp.iterrows():
    G.add_node(
        row['Component ID'],
        label=row['Component Name'],
        subsystem=row['Subsystem'],
        duration=row['Duration (hrs)'],
        is_critical=row['is_critical']
    )

# Tambah directed edges dari sheet Dependency_Edges
for _, row in df_edge.iterrows():
    src = row['FROM (Prerequisite ID)']
    dst = row['TO (Dependent ID)']
    weight = row['Edge Weight (hrs)']
    G.add_edge(src, dst, weight=weight)

print(f'Nodes : {G.number_of_nodes()}')
print(f'Edges : {G.number_of_edges()}')
print(f'Is DAG (acyclic): {nx.is_directed_acyclic_graph(G)}')

In [ ]:
# Cell 3 – Identifikasi root nodes & leaf nodes
root_nodes = [n for n, d in G.in_degree() if d == 0]
leaf_nodes  = [n for n, d in G.out_degree() if d == 0]

print(f'Root nodes ({len(root_nodes)}):')
for n in root_nodes:
    print(f'  {n} – {G.nodes[n]["label"]}')

print(f'\nLeaf nodes ({len(leaf_nodes)}):')
for n in leaf_nodes:
    print(f'  {n} – {G.nodes[n]["label"]}')

In [ ]:
# Cell 4 – Statistik DAG per subsystem
subsystem_counts = df_comp.groupby('Subsystem').agg(
    jumlah_komponen=('Component ID', 'count'),
    total_durasi=('Duration (hrs)', 'sum'),
    komponen_kritis=('is_critical', 'sum')
).sort_values('total_durasi', ascending=False)

print('Statistik per Subsystem:')
print(subsystem_counts.to_string())

In [ ]:
# Cell 5 – In-degree & out-degree distribution
in_deg  = dict(G.in_degree())
out_deg = dict(G.out_degree())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Distribusi Degree – Honda RA621H DAG', fontsize=13, fontweight='bold')

axes[0].hist(list(in_deg.values()), bins=range(0, max(in_deg.values())+2),
             color='#E10600', edgecolor='black', alpha=0.85)
axes[0].set_title('In-Degree Distribution')
axes[0].set_xlabel('In-Degree')
axes[0].set_ylabel('Jumlah Node')

axes[1].hist(list(out_deg.values()), bins=range(0, max(out_deg.values())+2),
             color='#1a1a1a', edgecolor='#E10600', alpha=0.85)
axes[1].set_title('Out-Degree Distribution')
axes[1].set_xlabel('Out-Degree')
axes[1].set_ylabel('Jumlah Node')

plt.tight_layout()
plt.savefig('dag_degree_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot tersimpan: dag_degree_distribution.png')

In [ ]:
# Cell 6 – Visualisasi DAG (subsystem view, spring layout)
# Warna tiap subsystem
subsystems = df_comp['Subsystem'].unique()
color_palette = [
    '#E10600', '#FF6B35', '#FFB347', '#4CAF50',
    '#2196F3', '#9C27B0', '#00BCD4', '#795548', '#607D8B'
]
subsystem_color = {s: color_palette[i % len(color_palette)] for i, s in enumerate(subsystems)}

node_colors = []
for n in G.nodes():
    sub = G.nodes[n].get('subsystem', '')
    node_colors.append(subsystem_color.get(sub, '#999999'))

# Node border merah tebal untuk critical path
node_edge_colors = ['#FFD700' if G.nodes[n]['is_critical'] else '#333333' for n in G.nodes()]
node_linewidths  = [2.5 if G.nodes[n]['is_critical'] else 0.5 for n in G.nodes()]

plt.figure(figsize=(22, 14))
plt.title('DAG – Honda RA621H Assembly (Spring Layout)', fontsize=15, fontweight='bold', pad=15)

pos = nx.spring_layout(G, seed=42, k=1.5)

nx.draw_networkx_nodes(G, pos,
                       node_color=node_colors,
                       node_size=300,
                       edgecolors=node_edge_colors,
                       linewidths=node_linewidths)

nx.draw_networkx_edges(G, pos,
                       edge_color='#555555',
                       arrows=True,
                       arrowsize=10,
                       width=0.6,
                       alpha=0.6,
                       connectionstyle='arc3,rad=0.05')

nx.draw_networkx_labels(G, pos,
                        labels={n: n for n in G.nodes()},
                        font_size=5,
                        font_color='white',
                        font_weight='bold')

# Legend subsystem
patches = [mpatches.Patch(color=subsystem_color[s], label=s.split('–')[-1].strip())
           for s in subsystems]
patches.append(mpatches.Patch(facecolor='gray', edgecolor='#FFD700',
                               linewidth=2.5, label='Critical Path'))
plt.legend(handles=patches, loc='upper left', fontsize=7, framealpha=0.85)

plt.axis('off')
plt.tight_layout()
plt.savefig('dag_full_graph.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print('Plot tersimpan: dag_full_graph.png')

In [ ]:
# Cell 7 – Ringkasan akhir DAG
critical_nodes = [n for n in G.nodes() if G.nodes[n]['is_critical']]
avg_in  = sum(dict(G.in_degree()).values())  / G.number_of_nodes()
avg_out = sum(dict(G.out_degree()).values()) / G.number_of_nodes()

print('=' * 45)
print('       RINGKASAN DAG – Honda RA621H')
print('=' * 45)
print(f'  Total nodes            : {G.number_of_nodes()}')
print(f'  Total edges            : {G.number_of_edges()}')
print(f'  Is acyclic (valid DAG) : {nx.is_directed_acyclic_graph(G)}')
print(f'  Root nodes             : {len(root_nodes)}')
print(f'  Leaf nodes             : {len(leaf_nodes)}')
print(f'  Critical path nodes    : {len(critical_nodes)}')
print(f'  Avg in-degree          : {avg_in:.2f}')
print(f'  Avg out-degree         : {avg_out:.2f}')
print('=' * 45)
print('DAG berhasil dibangun. Siap untuk Topological Sort & CPM.')